In [7]:
import gc

import pandas as pd
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from default_risk.scripts.auxiliars_for_modeling import apply_cyclical_encoding
import joblib
import lightgbm as lgb
from typing import Optional, List
import optuna
from optuna_integration.mlflow import MLflowCallback
import mlflow
import numpy as np
from sklearn.model_selection import cross_val_score
import xgboost as xgb



import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import get_pipeline

from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance
from default_risk.scripts.feature_cleaner import creating_criteria
from optuna_integration.mlflow import MLflowCallback

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
prev_app_df = pd.read_parquet(cfg.PROCESSED_DIR / "previous_application.train-processed.parquet")


merged_df = application_train_df.merge(
    prev_app_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_df
gc.collect()

X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)
run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train + prev_application")
#auc_score_OOF= 0.754

#freeing memory
del merged_df
gc.collect()



In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding")
prev_app_installment_agg_df = pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")


merged_df = application_train_df.merge(
    prev_app_installment_agg_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_installment_agg_df


gc.collect()


X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)




#run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"prev_app+installment fpi",enable_feature_permutation=True)

#importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_max_rows_internal_parent.csv")
#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "max_rows.csv")
#X = clean_noise_from_feature_importance(importance_df,X,0.0025)
#X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.0003)


run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"raw_internal_parent_target_encoding",persist_feature_importance=True)

#cleaned.to_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments")

#auc_score_OOF= 0.763

#freeing memory
del merged_df
gc.collect()

In [ ]:
run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"raw_internal_parent_target_encoding",persist_feature_importance=True)

#cleaned.to_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments")

#auc_score_OOF= 0.763

#freeing memory
del merged_df
gc.collect()

In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "pipeline_baseline.parquet")
prev_app_installment_agg_df = pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")


merged_df = application_train_df.merge(
    prev_app_installment_agg_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_installment_agg_df


gc.collect()


X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "max_cols_internal.csv")
importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_max_cols_internal.csv")

#X = clean_noise_from_feature_importance(importance_df,X,0.0025)
X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.00010)



run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"max_cols_internal")

#cleaned.to_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments")

#auc_score_OOF= 0.763

#freeing memory
del merged_df
gc.collect()

In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
prev_app_installment_agg_df = pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")


merged_df = application_train_df.merge(
    prev_app_installment_agg_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_installment_agg_df


gc.collect()


X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

X= apply_cyclical_encoding(X,"hour_appr_process_start_prev_1",24)




X.drop(columns=["hour_appr_process_start_prev_1"],inplace=True)

model= xgb.XGBClassifier(**hiperparams)
categorical_features= ["organization_type","occupation_type","code_reject_reason_prev_1","name_income_type","name_goods_category_prev_1","name_cash_loan_purpose_prev_1"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)#,,"product_combination_prev_1"

#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "internal_parent_target_enconding_max_cols_feature_importance.csv")
importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_internal_parent_target_enconding_max_cols.csv")

#X = clean_noise_from_feature_importance(importance_df,X,0.0024739875)
#X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.0003)

#pd.get_dummies(X,columns= ["name_contract_type"])



run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"internal_parent_target_enconding_max_cols")


#cleaned.to_parquet(cfg.PROCESSED_DIR / "pruned_prev_app_with_installments")

#auc_score_OOF= 0.763

#freeing memory
del merged_df
gc.collect()  

In [ ]:
#for the second one  we gonna analize the gains from the aggregation of bureau
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_feature_engineering.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train+bureau")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

In [ ]:
#now bureau parent (main with feature engineering + Bureau with feature engineering + Bureau_balance)
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "toxic_baseline.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)



#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_bureau_parent.csv")

#X= clean_importance_zero_and_negative_pfi(importance_df,X)
#importance_permutation_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_bureau_parent.csv")
importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "external_importance.csv")
#criteria = creating_criteria(importance_df,importance_permutation_df)

#importance2 = pd.read_csv(cfg.ARTIFACTS_DIR / "second filter.csv") #
#X= X.drop(columns=["bureau_balance_is_delincuency_sum_loan_1","bureau_has_bureau_balance_data_loan_1","ext_source_1_is_missing"]) 

X= clean_noise_from_feature_importance(importance_df,X,0.004)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"external_parent_co_sample_clean")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

In [ ]:
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

model= xgb.XGBClassifier(**hiperparams)
categorical_features=  ["organization_type","occupation_type","name_income_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)


#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_external_parent_target_encoding.csv")

#X= clean_importance_zero_and_negative_pfi(importance_df,X,0.0003)



run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"external_parent_target_encoding")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments_time_window.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()



X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

feature_raper= pd.read_csv(cfg.ARTIFACTS_DIR / "final_importance.csv")

X= clean_noise_from_feature_importance(feature_raper,X)

merged_df= merged_df.drop(columns=["flag_email"])

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"2.1 (app_train_with_features+bureau+prev_app+installments)")

#auc_score_OOF=  is the result of all the agregation at 2.0

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "pipeline_baseline.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)


#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")



merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()





X,Y = prepare_columns(merged_df)

X = cast_object_into_categoricals(X)

#importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_max_cols_internal.csv")




#X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.00010)


run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"max_rows_final_model")

#auc_score_OOF=  0.781

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

features_from_internal_historial= pd.read_csv(cfg.ARTIFACTS_DIR / "internal_best_result.csv")
features_from_external_historial= pd.read_csv(cfg.ARTIFACTS_DIR / "external_best_result.csv")

internal_list=  features_from_internal_historial["feature_name"].to_list()
external_list=  features_from_external_historial["feature_name"].to_list()
features_names = list(set(internal_list + external_list))

X,Y = prepare_columns(merged_df)

X= X[features_names]

X= X.drop(columns= ["amt_down_payment_sum"])



X = cast_object_into_categoricals(X)



run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"final_model_from_convination_of_best_results")

#auc_score_OOF=  0.781

#freeing memory
del merged_df
gc.collect()

In [6]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

par = {
'n_estimators': 2086,
'learning_rate': 0.010061666728286038,
'num_leaves': 68,
'min_child_samples': 151,
"max_depth" :-1,           
"subsample" : 0.8664564642957002,          
"colsample_bytree" : 0.6347840589896324,
'reg_alpha': 4.905930240549411,
'reg_lambda': 0.006830459057401382,
'min_split_gain': 0.25616917560909797,
'min_child_weight': 0.010819305760005674,   
"random_state" : 42,
"n_jobs" : -1,
"objective" : 'binary',
"force_col_wise": True,
"importance_type" : "gain"
}

best_params_optuna = {'n_estimators': 2086, 'learning_rate': 0.010061666728286038, 'num_leaves': 68, 'min_child_samples': 151, 'subsample': 0.8664564642957002, 'colsample_bytree': 0.6347840589896324, 'reg_alpha': 4.905930240549411, 'reg_lambda': 0.006830459057401382, 'min_split_gain': 0.25616917560909797, 'min_child_weight': 0.010819305760005674,"objective" : 'binary',
"force_col_wise": True,
"importance_type" : "gain","random_state" : 42}

model_lgbm = lgb.LGBMClassifier(**par)


categorical_features= ["organization_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1", "organization_type"


pipeline= get_pipeline(50,categorical_features,model_lgbm)





#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df,santize_text=True)



features_df= pd.read_csv(cfg.ARTIFACTS_DIR / "features_final_model.csv")

feature_list=  features_df["feature_name"].to_list()



feature_list = feature_list + ["amt_income_total"]#



X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)

features_to_target_encoding= ["instalment_income_ratio"]

X['random_noise'] = np.random.normal(0, 1, len(X))

X = cast_object_into_categoricals(X)

model=xgb.XGBClassifier(**hiperparams)

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm.csv")

X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.00001)

snd_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_with_first_cut.csv")

X = clean_importance_zero_and_negative_pfi(snd_filter,X,0.00001)


trd_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_with_third_cut.csv")

X = clean_importance_zero_and_negative_pfi(trd_filter,X,0.00007)



run_cv_tracked_mlflow(pipeline,par,cv,X,Y,experiment_name,"lightgbm_for_pipeline")

#auc_score_OOF=  0.781

#freeing memory
del merged_df
gc.collect()

eliminando ['hour_appr_process_start_prev_1', 'closed_log_amt_credit_sum_closed_mean', 'cnt_payment_max', 'amt_req_credit_breau_mon', 'credit_card_amt_credit_limit_actual_std_prev_1', 'amt_credit_max', 'instalments_amount_of_versions_in_sequence_sum', 'active_credit_type_credit_card_active_sum', 'bureau_ratio_credit_annuity_loan_1', 'log_amt_down_payment_mean', 'active_balance_months_balance_min_active_min', 'housing_type', 'log_amt_down_payment_std', 'amt_application_sum', 'active_ratio_credit_annuity_active_mean', 'name_yield_group_prev_1', 'obs_60_cnt_social_circle', 'amt_goods_price_min', 'closed_balance_months_since_delincuency_closed_max', 'log_total_interest_charged_mean', 'credit_card_cnt_drawings_atm_current_mean_prev_1', 'active_ratio_credit_annuity_active_max', 'credit_card_name_contract_status_active_sum_prev_1', 'instalments_amt_instalment_median_prev_1', 'closed_amt_annuity_closed_mean', 'closed_balance_months_balance_min_closed_min', 'log_amt_credit_std', 'implied_intere

KeyboardInterrupt: 

In [ ]:
def eliminar_colinealidad(X: pd.DataFrame, umbral: float = 0.95, metodo: str = 'pearson') -> pd.DataFrame:

    X_num = X.select_dtypes(include=[np.number])
    
    corr_matrix = X_num.corr(method=metodo).abs()
    
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
   
    columnas_a_eliminar = [columna for columna in upper_tri.columns if any(upper_tri[columna] > umbral)]
    
    # 5. Retornar el DataFrame original sin esas columnas
    return columnas_a_eliminar

In [ ]:
def clean_colineality(X: pd.DataFrame, column_list) -> pd.DataFrame:

    return X.drop(columns=column_list)

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()
list_to_delete= eliminar_colinealidad(merged_df,0.95)




In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)

#X= clean_colineality(X,list_to_delete)


model= xgb.XGBClassifier(**hiperparams)
categorical_features= ["organization_type","occupation_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
 #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)


X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)

X = cast_object_into_categoricals(X)



#X= X.drop(columns=cols_to_drop)

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_monster_final_model.csv")

X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,-0.00001)

second_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "second.csv")

X = clean_importance_zero_and_negative_pfi(second_filter,X,-0.00001)

third_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "third.csv")

X = clean_importance_zero_and_negative_pfi(third_filter,X,-0.00001)

fourth_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "fourth.csv")

X = clean_importance_zero_and_negative_pfi(fourth_filter,X,0.00001)



run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"monster_without_co_lineality",enable_feature_permutation=False)

#auc_score_OOF=  0.781

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)

#X= clean_colineality(X,list_to_delete)




model= xgb.XGBClassifier(**hiperparams)
categorical_features= ["organization_type","occupation_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
 #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)


X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)

X = cast_object_into_categoricals(X)

cols_to_drop= ["name_income_type"]

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_monster_final_model.csv")


X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,-0.00001)

second_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "second.csv")

X = clean_importance_zero_and_negative_pfi(second_filter,X,-0.00001)

third_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "third.csv")

X = clean_importance_zero_and_negative_pfi(third_filter,X,-0.00001)

fourth_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "fourth.csv")

X = clean_importance_zero_and_negative_pfi(fourth_filter,X,0.00001)

X= X.drop(columns=cols_to_drop)






#five_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_fine_pruned_final_model.csv")

#X = clean_importance_zero_and_negative_pfi(five_filter,X,0.00009)

run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"fine_pruned_final_model")


In [ ]:
# 1. Función Objetivo (Ahora recibe X, Y, y las categóricas explícitamente)
mlflow_callback = MLflowCallback(
    tracking_uri="mlruns",
    metric_name="roc_auc",
    create_experiment=True
)

def objective(trial, X, Y, categorical_features):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 5000),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),

        "num_leaves": trial.suggest_int("num_leaves", 16, 256, log=True),

        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),

        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),

        "min_child_samples" : trial.suggest_int("min_child_samples",20,300),

        "min_split_gain": trial.suggest_float(
        "min_split_gain", 0.0, 1.0
        ),

        "min_child_weight": trial.suggest_float(
        "min_child_weight", 1e-3, 10.0, log=True
        ),

        # --- Parámetros fijos ---
        "max_depth": -1, 
        "random_state": 42,
        "n_jobs" : 14,
        "objective": 'binary',      # En LightGBM sí se llama así
        "force_col_wise": True,
        "importance_type": "gain"
        # OJO: Quitamos "enable_categorical" porque en LightGBM se configura distinto
    }

    model = lgb.LGBMClassifier(**params)
    pipeline = get_pipeline(50, categorical_features, model)
    
    auc_scores = cross_val_score(
        pipeline, 
        X, 
        Y, 
        cv=5, 
        scoring="roc_auc", 
        n_jobs=1
    )
    
    return np.mean(auc_scores)


# 2. Función Principal (Recibe los datos y configura el estudio)
def run_optimization(X_train, Y_train, cat_features):
    study = optuna.create_study(
        study_name="lightgbm_tuning",
        direction="maximize" 
    )
    
    # EL TRUCO: Usamos un lambda para inyectar los datos preservando el 'trial'
    study.optimize(
        lambda trial: objective(trial, X_train, Y_train, cat_features), 
        n_trials=50, 
        callbacks=[mlflow_callback]
    )
    
    return study


    

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)


categorical_features= ["organization_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1", "organization_type"

#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df,santize_text=True)



features_df= pd.read_csv(cfg.ARTIFACTS_DIR / "features_final_model.csv")

feature_list=  features_df["feature_name"].to_list()



feature_list = feature_list + ["amt_income_total"]#



X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)

features_to_target_encoding= ["instalment_income_ratio"]

X['random_noise'] = np.random.normal(0, 1, len(X))

X = cast_object_into_categoricals(X)

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm.csv")

X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.00001)

snd_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_with_first_cut.csv")

X = clean_importance_zero_and_negative_pfi(snd_filter,X,0.00001)


trd_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_with_third_cut.csv")

X = clean_importance_zero_and_negative_pfi(trd_filter,X,0.00007)



study = run_optimization(X,Y,categorical_features)
print(f"Mejor AUC alcanzado: {study.best_value}")
print(f"Mejores Hiperparámetros: {study.best_params}")

In [ ]:
test_application= pd.read_parquet(cfg.MASTER_DATA_DIR / "prepared_dataset_test.parquet")
dtale.show(test_application.head(5))

Mejores Hiperparámetros: {'n_estimators': 2086, 'learning_rate': 0.010061666728286038, 'num_leaves': 68, 'min_child_samples': 151, 'subsample': 0.8664564642957002, 'colsample_bytree': 0.6347840589896324, 'reg_alpha': 4.905930240549411, 'reg_lambda': 0.006830459057401382, 'min_split_gain': 0.25616917560909797, 'min_child_weight': 0.010819305760005674}


In [ ]:
internal_list = [
"applications_count",
"instalments_amount_of_versions_in_sequence_prev_1",
"instalments_days_of_delinquency_max_max",
"name_contract_type",
"last_365_instalments_completion_ratio",
"days_termination_prev_1",
"payment_trend",
"amt_goods_price_max",
"days_last_due_1st_version_prev_1",
"organization_type_Construction",
"ratio_credit_to_goods_max",
"organization_type_Self-employed",
"diff_application_credit_max",
"implied_interest_rate_std",
"last_6_cash_balance_amount_advanced_payment_max",
"last_6_credit_card_balance_limit_ratio_mean",
"amt_req_credit_breau_qrt",
"building_score_sum",
"name_seller_industry_prev_1",
"days_decision_min",
"implied_interest_rate_mean",
"def_60_cnt_social_circle",
"days_decision_mean",
"ratio_debt_age",
"instalments_days_of_underpayment_max_max",
"last_365_instalments_amt_payment_min",
"instalments_amt_instalment_sum_sum",
"last_90_instalments_completion_ratio",
"amt_annuity",
"instalments_completion_ratio_mean",
"last_6_cash_balance_count_instalment_future_min",
"days_employed",
"amt_credit",
"instalments_days_of_delinquency_mean_mean",
"ratio_credit_to_goods_mean",
"family_status",
"days_birth",
"ext_source_3",
"days_and_insurance_information_are_missing_mean",
"instalments_days_of_delinquency_mean_prev_1",
"credit_card_is_over_the_limit_mean_mean",
"ext_source_1",
"ext_1_x_2",
"kui_ratio",
"credit_card_cnt_drawings_atm_current_mean_mean",
"last_6_cash_balance_count_instalment_future_max",
"last_365_instalments_is_delinquency_mean",
"amt_down_payment_sum",
"name_income_type",
"documents_count",
"amt_req_credit_breau_day",
"amt_goods_price",
"region_raiting_client_city",
"last_365_instalments_extra_instalament_mean",
"last_6_credit_card_completion_ratio",
"flag_not_live_city",
"instalments_is_delinquency_mean_mean",
"own_car_age",
"code_reject_reason_prev_1",
"def_30_cnt_social_circle",
"instalments_amt_payment_sum_sum",
"instalments_extra_instalament_mean_mean",
"occupation_type",
"credit_duration",
"implied_interest_rate_max",
"last_365_instalments_days_of_delinquency_mean",
"last_6_cash_balance_sk_dpd_def_sum",
"education_type_Secondary / secondary special",
"ratio_good_credit",
"region_raiting_client",
"code_gender",
"last_6_cash_balance_sk_dpd_mean",
"flag_document_3",
"last_6_credit_card_balance_limit_ratio_max",
"ext_2_x_3",
"education_type_Higher education",
"ext_source_mean"
]


external_list= [
"ext_source_1_is_missing",
"amt_req_credit_breau_day",
"organization_type_Bank",
"region_raiting_client",
"closed_ratio_credit_annuity_closed_max",
"organization_type_Construction",
"active_amt_annuity_active_std",
"bureau_days_enddate_fact_loan_1",
"active_have_amt_credit_sum_overdue_active_sum",
"flag_own_realty",
"closed_days_credit_update_closed_max",
"building_score_sum",
"closed_amt_credit_sum_debt_closed_mean",
"active_amt_credit_max_overdue_active_max",
"closed_days_credit_update_closed_min",
"bureau_balance_is_delincuency_mean_loan_1",
"active_amt_annuity_is_missing_active_sum",
"active_days_credit_enddate_active_mean",
"active_amt_credit_sum_active_max",
"closed_amt_credit_sum_debt_closed_std",
"bureau_days_credit_enddate_loan_1",
"closed_amt_credit_sum_closed_max",
"organization_type_Military",
"building_score_mean",
"amt_req_credit_breau_qrt",
"bureau_balance_is_delincuency_sum_loan_1",
"days_id_publish",
"ratio_debt_age",
"wallsmaterial_mode",
"closed_amt_credit_max_overdue_closed_mean",
"bureau_balance_status_score_mean_loan_1",
"ratio_days_employed_days_lived",
"organization_type_Self-employed",
"closed_amt_credit_sum_closed_sum",
"kui_ratio",
"ext_1_x_3",
"organization_type_Transport: type 3",
"active_amt_credit_max_overdue_active_sum",
"days_birth",
"amt_credit",
"def_60_cnt_social_circle",
"bureau_amt_credit_max_overdue_loan_1",
"name_contract_type",
"ext_source_1",
"name_income_type",
"amt_annuity",
"active_id_curr_active_count",
"days_employed",
"bureau_amt_credit_sum_debt_is_missing_loan_1",
"amt_goods_price",
"family_status",
"bureau_completetitud_ratio_loan_1",
"active_completetitud_ratio_active_mean",
"occupation_type",
"region_raiting_client_city",
"bureau_credit_type_loan_1",
"ext_2_x_3",
"ext_1_x_2",
"own_car_age",
"flag_not_live_city",
"active_completetitud_ratio_active_min",
"credit_duration",
"education_type_Secondary / secondary special",
"def_30_cnt_social_circle",
"documents_count",
"flag_document_3",
"code_gender",
"ratio_good_credit",
"ext_source_3",
"ext_source_2",
"education_type_Higher education",
"ext_source_mean"]

features_names = list(set(internal_list + external_list))

In [3]:
print(features_names)

['bureau_balance_is_delincuency_sum_loan_1', 'closed_days_credit_update_closed_max', 'instalments_completion_ratio_mean', 'last_365_instalments_days_of_delinquency_mean', 'implied_interest_rate_mean', 'last_365_instalments_is_delinquency_mean', 'active_amt_credit_sum_active_max', 'ratio_credit_to_goods_max', 'ext_source_2', 'active_completetitud_ratio_active_min', 'closed_amt_credit_sum_debt_closed_mean', 'payment_trend', 'active_id_curr_active_count', 'wallsmaterial_mode', 'flag_document_3', 'code_reject_reason_prev_1', 'ext_2_x_3', 'bureau_balance_status_score_mean_loan_1', 'active_amt_annuity_active_std', 'ext_source_3', 'region_raiting_client_city', 'amt_goods_price_max', 'amt_down_payment_sum', 'days_and_insurance_information_are_missing_mean', 'education_type_Secondary / secondary special', 'amt_credit', 'closed_days_credit_update_closed_min', 'name_income_type', 'days_termination_prev_1', 'building_score_mean', 'closed_ratio_credit_annuity_closed_max', 'code_gender', 'bureau_day

In [8]:
features_gbm= pd.read_csv(cfg.ARTIFACTS_DIR / "lightgbm_for_pipeline_feature_importance.csv")
name_list = features_gbm["feature_name"].to_list()
print(name_list)

['credit_card_cnt_drawings_current_max_prev_1', 'name_type_suite_prev_1', 'active_credit_type_microloan_active_mean', 'amt_req_credit_breau_qrt', 'instalments_amount_of_versions_in_sequence_max', 'active_credit_type_mortgage_active_mean', 'last_6_cash_balance_amount_advanced_payment_sum', 'code_reject_reason_prev_1', 'organization_type', 'bureau_credit_type_loan_1', 'education_type', 'name_contract_status_canceled_mean', 'credit_card_cnt_instalment_mature_cum_max_max', 'active_amt_annuity_is_missing_active_sum', 'instalments_is_delinquency_sum_sum', 'rate_down_payment_prev_1', 'active_amt_credit_max_overdue_active_sum', 'instalments_days_of_delinquency_max_prev_1', 'closed_id_curr_closed_count', 'documents_count', 'last_6_credit_card_desesperation_ratio_mean', 'active_id_curr_active_count', 'wallsmaterial_mode', 'flag_document_3', 'closed_balance_is_delincuency_mean_closed_mean', 'def_30_cnt_social_circle', 'active_credit_type_consumer_credit_active_sum', 'credit_card_amt_balance_max_m